# 🧪 Praktikum 07 – Vektordatenbanken & Profi-RAG
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Persistente Vektorspeicher verwalten · Hybride Retrieval-Strategien · RAG-Antworten auf Faktentreue (Faithfulness) prüfen

⏱️ **Dauer:** 90 Minuten

In [ ]:
import os, sys, subprocess, shutil, time, requests
from importlib.util import find_spec
MODEL = "qwen3.5:0.8b"
EMBED_MODEL = "nomic-embed-text"
def install_if_missing(pkgs):
    to_install = [p for p in pkgs if find_spec(p) is None]
    if to_install: subprocess.check_call([sys.executable, "-m", "pip", "install"] + (["--break-system-packages"] if sys.prefix == getattr(sys, "base_prefix", sys.prefix) else []) + to_install)
install_if_missing(["chromadb", "ollama", "requests"])
import chromadb, ollama
def setup_ollama():
    try: requests.get("http://localhost:11434/api/tags", timeout=2)
    except: subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); time.sleep(5)
    for m in [MODEL, EMBED_MODEL]: subprocess.check_call(["ollama", "pull", m])
setup_ollama()
print("✅ Setup abgeschlossen.")

## Teil 1 – Advanced ChromaDB Management ⏱️ 30 min
Wir nutzen Metadaten-Filter und persistente Indexierung.

In [ ]:
client = chromadb.PersistentClient(path="./rag_storage")
collection = client.get_or_create_collection(name="docs")

texts = [
    "Python ist eine interpretierte Hochsprache.",
    "C++ ist eine kompilierte, maschinennahe Sprache.",
    "Rust bietet Speichersicherheit ohne Garbage Collector.",
    "Java nutzt eine Virtual Machine (JVM)."
]

def embed(t): return ollama.embeddings(model=EMBED_MODEL, prompt=t)["embedding"]

if collection.count() == 0:
    collection.add(
        documents=texts,
        embeddings=[embed(t) for t in texts],
        metadatas=[{"lang_type": "high"}, {"lang_type": "low"}, {"lang_type": "safe"}, {"lang_type": "vm"}],
        ids=[f"id{i}" for i in range(len(texts))]
    )
print(f"Anzahl Dokumente in DB: {collection.count()}")

## Teil 2 – Hybrides Retrieval ⏱️ 30 min
Nur Vektorsuche reicht oft nicht. Wir kombinieren sie mit Keyword-Matching (simuliert).

In [ ]:
def hybrid_search(query, top_k=2):
    # 1. Vektor-Suche
    v_res = collection.query(query_embeddings=[embed(query)], n_results=top_k)
    # 2. Keyword-Filter (Simuliert BM25)
    keywords = query.lower().split()
    final_docs = []
    for doc in v_res['documents'][0]:
        score = sum(1 for kw in keywords if kw in doc.lower())
        final_docs.append((score, doc))
    
    # Ranking nach Keyword-Treffern, dann Vektor-Relevanz
    final_docs.sort(key=lambda x: x[0], reverse=True)
    return [d[1] for d in final_docs]

print("Hybride Suche für 'JVM Sprache':", hybrid_search("JVM Sprache"))

## Teil 3 – RAG Faithfulness & Self-Correction ⏱️ 30 min
Wir lassen das Modell prüfen, ob die Antwort wirklich im Kontext steht.

In [ ]:
def rag_with_check(query):
    context = "\n".join(hybrid_search(query))
    gen_prompt = f"Antworte auf Basis des Kontextes: {context}\nFrage: {query}"
    answer = ollama.chat(model=MODEL, messages=[{"role": "user", "content": gen_prompt}])['message']['content']
    
    check_prompt = f"Steht die folgende Aussage im Text?\nText: {context}\nAussage: {answer}\nAntworte nur mit JA oder NEIN."
    valid = ollama.chat(model=MODEL, messages=[{"role": "user", "content": check_prompt}])['message']['content']
    
    return answer, valid

ans, status = rag_with_check("Was ist Rust?")
print(f"Antwort: {ans}\nVerifiziert: {status}")

## 🧩 Aufgaben
1. **Re-Ranking:** Implementiere ein einfaches Re-Ranking, das Dokumente bevorzugt, die kürzer als 50 Zeichen sind.
2. **Error Handling:** Was passiert, wenn kein Dokument gefunden wird? Baue einen Fallback-Prompt.
3. **Metadaten-Explosion:** Füge 100 Test-Dokumente hinzu und filtere nach `lang_type`. Messe die Zeitdifferenz zur ungefilterten Suche.